# Notebook 2 — Data Processing

**Purpose:** Transform raw TravisTorrent data into a clean, model-ready dataset. Feature engineering is based on the literature (Kumar et al., 2025): build duration, commit count, lines changed, temporal features, test pass rate, and previous-build-failed lag. We handle missing values explicitly and save train/validation/test splits plus encoders for downstream notebooks and the dashboard.

In [40]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import joblib

# Project root and paths
ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))
from src.preprocessing import process_raw_to_features, fill_and_encode, get_feature_columns
from src.paths import DATA_RAW, DATA_PROCESSED, MODELS_DIR

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

### Load raw data

Load the TravisTorrent CSV. We check for the presence of columns before using them, so different file versions (e.g. from Zenodo) remain supported.

In [41]:
raw_path = DATA_RAW / "final-2017-01-25.csv"
if not raw_path.exists():
    raw_path = ROOT / "data" / "raw" / "final-2017-01-25.csv"
# Full dataset for 95%+ accuracy push; if memory-constrained use nrows=1_000_000
nrows = None  # None = load all rows
print("Loading raw CSV (full dataset; may take 1–2 min)...")
df = pd.read_csv(raw_path, nrows=nrows, low_memory=False)
print("Load complete.")
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")
print("Target tr_status:", df["tr_status"].value_counts().to_dict() if "tr_status" in df.columns else "MISSING")

Loading raw CSV (full dataset; may take 1–2 min)...
Load complete.
Loaded 3,881,992 rows × 66 columns
Target tr_status: {'passed': 2810783, 'failed': 751531, 'errored': 279483, 'canceled': 40192, 'started': 3}


### Feature engineering

Each step checks whether the source column exists and prints a message if it is missing. This keeps the pipeline robust across TravisTorrent versions.

In [42]:
print("Running feature engineering (duration, commits, churn, temporal, test pass rate, previous_build_failed)...")
X_eng, y = process_raw_to_features(df, target_col="tr_status")
print("Feature engineering complete.")
print("Engineered features:", list(X_eng.columns))
print("Target shape:", y.shape if y is not None else None)

Running feature engineering (duration, commits, churn, temporal, test pass rate, previous_build_failed)...
Feature engineering complete.
Engineered features: ['build_duration_sec', 'commits_per_build', 'loc_changed', 'hour_of_day', 'day_of_week', 'is_weekend', 'test_pass_rate', 'previous_build_failed']
Target shape: (3881992,)


### Keep only the four main target classes

The full dataset can include rare statuses (e.g. "started" with very few rows). Stratified split requires at least 2 samples per class in each split, so we keep only **passed**, **failed**, **errored**, and **canceled**.

In [43]:
# Keep only the four classes required for stratified split (each class needs >= 2 samples in every split)
ALLOWED_STATUS = ["passed", "failed", "errored", "canceled"]
keep = y.isin(ALLOWED_STATUS)
n_dropped = (~keep).sum()
y = y[keep].reset_index(drop=True)
X_eng = X_eng.loc[keep].reset_index(drop=True)
df = df.loc[keep].reset_index(drop=True)  # keep df in sync so gh_lang aligns
if n_dropped > 0:
    print(f"Dropped {n_dropped:,} rows with target not in {ALLOWED_STATUS} (e.g. 'started'). Kept {len(y):,} rows.")
else:
    print(f"All {len(y):,} rows have target in {ALLOWED_STATUS}.")

Dropped 3 rows with target not in ['passed', 'failed', 'errored', 'canceled'] (e.g. 'started'). Kept 3,881,989 rows.


### Add programming language (categorical)

We add `gh_lang` from the raw data for label encoding. If the column is missing, we skip it and the encoder will not be used.

In [44]:
if "gh_lang" in df.columns:
    X_eng["gh_lang"] = df["gh_lang"].astype(str).fillna("unknown")
    print("Added gh_lang for encoding.")
else:
    print("gh_lang not found in raw data; skipping.")

Added gh_lang for encoding.


### Handle missing values and encode target

Numeric nulls are filled with the column median. Categorical columns (e.g. `gh_lang`) are label-encoded. The target is encoded from string labels to integers and the mapping is stored so we can reverse it later (e.g. in the dashboard).

In [45]:
print("Filling missing values (median for numeric, label encoding for categorical) and encoding target...")
X_final, y_enc, target_encoder, lang_encoder = fill_and_encode(X_eng, y, cat_columns=[])
print("Fill and encode complete.")
feature_list = [c for c in get_feature_columns() if c in X_final.columns]
X_final = X_final[feature_list]
class_names = list(target_encoder.classes_)
print("Feature list:", X_final.columns.tolist())
print("Class names (target mapping):", class_names)

Filling missing values (median for numeric, label encoding for categorical) and encoding target...
Fill and encode complete.
Feature list: ['build_duration_sec', 'commits_per_build', 'loc_changed', 'hour_of_day', 'day_of_week', 'is_weekend', 'test_pass_rate', 'previous_build_failed', 'gh_lang_enc']
Class names (target mapping): ['canceled', 'errored', 'failed', 'passed']


### Stratified train / validation / test split

Split 70% train, 15% validation, 15% test, stratified by the target to preserve class proportions in each set.

In [46]:
print("Splitting into train (70%) / validation (15%) / test (15%), stratified by target...")
X_train, X_rest, y_train, y_rest = train_test_split(
    X_final, y_enc, test_size=0.30, stratify=y_enc, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=42
)
print("Split complete.")
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Splitting into train (70%) / validation (15%) / test (15%), stratified by target...
Split complete.
Train: 2,717,392 | Val: 582,298 | Test: 582,299


### Save splits and encoders

All artifacts are saved so Notebook 3 (training), Notebook 4 (SHAP), and the dashboard can load them without re-running processing.

In [47]:
print("Saving train/val/test splits and encoders to disk...")
X_train.to_csv(DATA_PROCESSED / "X_train.csv", index=False)
X_val.to_csv(DATA_PROCESSED / "X_val.csv", index=False)
X_test.to_csv(DATA_PROCESSED / "X_test.csv", index=False)
np.save(DATA_PROCESSED / "y_train.npy", y_train)
np.save(DATA_PROCESSED / "y_val.npy", y_val)
np.save(DATA_PROCESSED / "y_test.npy", y_test)

encoders = {
    "target_encoder": target_encoder,
    "lang_encoder": lang_encoder,
    "feature_list": X_train.columns.tolist(),
    "class_names": class_names,
}
joblib.dump(encoders, MODELS_DIR / "encoders.joblib")
print("Save complete: X_train, X_val, X_test, y_*.npy, encoders.joblib.")

Saving train/val/test splits and encoders to disk...
Save complete: X_train, X_val, X_test, y_*.npy, encoders.joblib.
